# 🚀 Introduction to Kubernetes
### Assignment Guide — Google Colab Edition

---

This notebook walks you through all four parts of the Kubernetes assignment **directly inside Google Colab**.

We use **`kind`** (Kubernetes IN Docker) instead of Minikube because Docker is already available in Colab and `kind` is lightweight and reliable in this environment.

| Part | Topic |
|------|-------|
| 1 | Kubernetes Setup |
| 2 | Application Deployment |
| 3 | Resource Management |
| 4 | Helm Charts |

> ⚠️ **Important:** Run cells **in order, one at a time**. Each step depends on the previous one.

---
## Part 1 — Kubernetes Setup

### What is Kubernetes?
Kubernetes (K8s) is an open-source **container orchestration platform**. It automates:
- Deploying containerized apps
- Scaling them up or down
- Restarting them if they crash
- Load-balancing traffic across them

### What is `kind`?
`kind` = **K**ubernetes **IN** **D**ocker. It runs a full Kubernetes cluster inside Docker containers — no VM needed. Perfect for Colab.

### What is `kubectl`?
`kubectl` is the **command-line tool** you use to talk to a Kubernetes cluster — like `git` is to GitHub.

In [ ]:
# Step 1: Install kubectl (the Kubernetes CLI)
# We download the latest stable binary and place it in /usr/local/bin

!curl -LO "https://dl.k8s.io/release/$(curl -Ls https://dl.k8s.io/release/stable.txt)/bin/linux/amd64/kubectl"
!chmod +x kubectl
!mv kubectl /usr/local/bin/kubectl
!kubectl version --client

In [ ]:
# Step 2: Install kind (Kubernetes in Docker)
# kind lets us run a real K8s cluster inside Docker containers — no VM required

!curl -Lo /usr/local/bin/kind \
  https://kind.sigs.k8s.io/dl/v0.22.0/kind-linux-amd64
!chmod +x /usr/local/bin/kind
!kind version

In [ ]:
# Step 3: Create a local Kubernetes cluster
# 'kind create cluster' spins up a single-node K8s cluster inside Docker
# --name gives our cluster a label we can reference later

!kind create cluster --name k8s-assignment

# This may take 1-2 minutes. You'll see:
#  ✓ Ensuring node image
#  ✓ Preparing nodes
#  ✓ Writing configuration
#  ✓ Starting control-plane
#  ✓ Installing CNI
#  ✓ Installing StorageClass
# Set kubectl context to "kind-k8s-assignment"

In [ ]:
# Step 4: Verify the cluster is running
# 'cluster-info' shows the control plane URL
# 'get nodes' lists all nodes — you should see STATUS = Ready

!kubectl cluster-info --context kind-k8s-assignment
print("\n--- Nodes ---")
!kubectl get nodes

✅ **Expected output from `kubectl get nodes`:**
```
NAME                          STATUS   ROLES           AGE   VERSION
k8s-assignment-control-plane  Ready    control-plane   60s   v1.29.x
```
If STATUS says `Ready`, your cluster is up and working!

---
## Part 2 — Application Deployment

### Key Concepts

| Term | What it means |
|------|---------------|
| **Pod** | The smallest unit in K8s — wraps one or more containers |
| **Deployment** | Manages Pods — ensures N replicas are always running |
| **Service** | Exposes Pods to network traffic (internal or external) |
| **Manifest** | A YAML file describing what you want K8s to create |

We'll deploy **nginx** (a popular web server) as our sample application.

In [ ]:
# Step 1: Write the Deployment manifest to a YAML file
# This file tells Kubernetes:
#   - What image to run (nginx:latest)
#   - How many copies to keep running (replicas: 2)
#   - Which port the container listens on (80)

deployment_yaml = """
apiVersion: apps/v1
kind: Deployment
metadata:
  name: nginx-deployment
  labels:
    app: nginx
spec:
  replicas: 2
  selector:
    matchLabels:
      app: nginx
  template:
    metadata:
      labels:
        app: nginx
    spec:
      containers:
      - name: nginx
        image: nginx:latest
        ports:
        - containerPort: 80
"""

with open('nginx-deployment.yaml', 'w') as f:
    f.write(deployment_yaml)

print("✅ nginx-deployment.yaml created successfully")
print("\n--- File Contents ---")
!cat nginx-deployment.yaml

In [ ]:
# Step 2: Apply the manifest to the cluster
# 'kubectl apply -f' reads the YAML and creates the resources

!kubectl apply -f nginx-deployment.yaml

In [ ]:
# Step 3: Wait for pods to be Running
# Kubernetes needs to pull the nginx image from Docker Hub first
# --timeout=90s waits up to 90 seconds for the rollout to finish

!kubectl rollout status deployment/nginx-deployment --timeout=90s

print("\n--- Deployments ---")
!kubectl get deployments

print("\n--- Pods ---")
!kubectl get pods

✅ **Expected output from `kubectl get pods`:**
```
NAME                                READY   STATUS    RESTARTS   AGE
nginx-deployment-xxxxxxxxxx-xxxxx   1/1     Running   0          30s
nginx-deployment-xxxxxxxxxx-xxxxx   1/1     Running   0          30s
```
Two pods, both `Running` with `1/1` ready — perfect!

In [ ]:
# Step 4: Create a Service manifest to expose the Deployment
# A Service gives your Pods a stable network address.
# ClusterIP = accessible only inside the cluster (suitable for Colab)

service_yaml = """
apiVersion: v1
kind: Service
metadata:
  name: nginx-service
spec:
  selector:
    app: nginx
  ports:
    - protocol: TCP
      port: 80
      targetPort: 80
  type: ClusterIP
"""

with open('nginx-service.yaml', 'w') as f:
    f.write(service_yaml)

!kubectl apply -f nginx-service.yaml

print("\n--- Services ---")
!kubectl get services

In [ ]:
# Step 5: Test the app by reaching nginx from inside the cluster
# We exec into a pod and curl the service's ClusterIP

import subprocess

# Get ClusterIP of the service
result = subprocess.run(
    ['kubectl', 'get', 'service', 'nginx-service',
     '-o', 'jsonpath={.spec.clusterIP}'],
    capture_output=True, text=True
)
cluster_ip = result.stdout.strip()
print(f"Service ClusterIP: {cluster_ip}")

# Get first pod name
pod_result = subprocess.run(
    ['kubectl', 'get', 'pods', '-o', 'jsonpath={.items[0].metadata.name}'],
    capture_output=True, text=True
)
pod_name = pod_result.stdout.strip()
print(f"Testing from pod: {pod_name}")

# curl nginx from inside the pod
!kubectl exec {pod_name} -- curl -s --max-time 5 http://{cluster_ip} | head -5

---
## Part 3 — Resource Management

This part covers the essential `kubectl` commands for managing live Kubernetes resources:

| Command | What it does |
|---------|-------------|
| `kubectl get` | List resources |
| `kubectl describe` | Detailed info about a resource |
| `kubectl logs` | View container logs |
| `kubectl scale` | Change replica count |
| `kubectl delete` | Remove a resource |

In [ ]:
# --- PODS ---

# List all pods with extra details (-o wide shows node + IP)
print("=== All Pods (wide) ===")
!kubectl get pods -o wide

In [ ]:
# Describe a pod — shows full config, events, and resource limits
# This is the most useful debugging command in Kubernetes

import subprocess
pod = subprocess.run(
    ['kubectl','get','pods','-o','jsonpath={.items[0].metadata.name}'],
    capture_output=True, text=True
).stdout.strip()

print(f"Describing pod: {pod}\n")
!kubectl describe pod {pod}

In [ ]:
# View logs from a pod
# Logs show stdout/stderr output from the container
# --tail=20 shows the last 20 lines

print(f"Logs from: {pod}\n")
!kubectl logs {pod} --tail=20

In [ ]:
# --- DEPLOYMENTS ---

# View all deployments and their status
print("=== Deployments ===")
!kubectl get deployments

# Describe the deployment — shows update strategy, selector, and events
print("\n=== Describe Deployment ===")
!kubectl describe deployment nginx-deployment

In [ ]:
# Scale the deployment — change from 2 replicas to 4
# Kubernetes will spin up 2 more pods automatically

print("Scaling nginx-deployment to 4 replicas...")
!kubectl scale deployment nginx-deployment --replicas=4

# Wait for the scale-up to finish
!kubectl rollout status deployment/nginx-deployment --timeout=60s

print("\n=== Pods after scaling ===")
!kubectl get pods

In [ ]:
# Scale back down to 2
# Kubernetes gracefully terminates the extra pods

!kubectl scale deployment nginx-deployment --replicas=2
!kubectl rollout status deployment/nginx-deployment --timeout=60s

print("\n=== Pods after scale-down ===")
!kubectl get pods

In [ ]:
# --- SERVICES ---

print("=== All Services ===")
!kubectl get services

print("\n=== Describe Service ===")
!kubectl describe service nginx-service

In [ ]:
# --- EVENTS ---
# Cluster events show everything that's happened (image pulls, pod starts, etc.)
# Very useful for debugging when something goes wrong

print("=== Recent Cluster Events ===")
!kubectl get events --sort-by='.lastTimestamp' | tail -20

In [ ]:
# --- DELETE a resource ---
# Deleting a pod controlled by a Deployment is safe —
# Kubernetes will automatically recreate it to maintain replica count

import subprocess, time

pod_to_delete = subprocess.run(
    ['kubectl','get','pods','-o','jsonpath={.items[0].metadata.name}'],
    capture_output=True, text=True
).stdout.strip()

print(f"Deleting pod: {pod_to_delete}")
!kubectl delete pod {pod_to_delete}

time.sleep(3)
print("\nPods after deletion (K8s recreated a replacement):")
!kubectl get pods

---
## Part 4 — Helm Charts

### What is Helm?
Helm is the **package manager for Kubernetes** — like `apt` for Ubuntu or `pip` for Python.

Without Helm, deploying a complex app means maintaining many separate YAML files. Helm bundles them into one **chart** and lets you:
- Install, upgrade, and rollback releases with one command
- Customize deployments with a `values.yaml` file
- Share and reuse charts via repositories

### Key Terms
| Term | Meaning |
|------|---------|
| **Chart** | A package of K8s YAML templates |
| **Release** | A running instance of a chart |
| **Repository** | A collection of charts (like a package registry) |
| **values.yaml** | Config file to customise a chart's defaults |

In [ ]:
# Step 1: Install Helm 3
# The official install script downloads the latest Helm binary

!curl https://raw.githubusercontent.com/helm/helm/main/scripts/get-helm-3 | bash
!helm version

In [ ]:
# Step 2: Add the Bitnami chart repository
# Bitnami provides production-ready charts for popular apps

!helm repo add bitnami https://charts.bitnami.com/bitnami
!helm repo update

print("\n=== Available Repos ===")
!helm repo list

In [ ]:
# Step 3: Search for available charts in the repo

print("=== Search for nginx charts ===")
!helm search repo nginx

print("\n=== Search for apache charts ===")
!helm search repo apache

In [ ]:
# Step 4: Install a Helm release
# 'helm install <release-name> <chart>' deploys the chart to the cluster
# --set lets you override default values inline
# We set service.type=ClusterIP so it works in the Colab environment

!helm install my-nginx bitnami/nginx \
  --set service.type=ClusterIP \
  --set replicaCount=1

print("\n=== Helm Releases ===")
!helm list

In [ ]:
# Wait for the Helm-deployed pod to be ready

!kubectl rollout status deployment/my-nginx --timeout=120s

print("\n=== All Pods (Helm release + our Deployment) ===")
!kubectl get pods

In [ ]:
# Step 5: Inspect the release — show all K8s objects Helm created

print("=== Helm Release Status ===")
!helm status my-nginx

print("\n=== Resources created by this Helm release ===")
!kubectl get all -l app.kubernetes.io/instance=my-nginx

In [ ]:
# Step 6: Upgrade a release
# helm upgrade lets you change config without reinstalling
# Here we bump the replica count from 1 → 2

!helm upgrade my-nginx bitnami/nginx \
  --set service.type=ClusterIP \
  --set replicaCount=2

!kubectl rollout status deployment/my-nginx --timeout=60s

print("\n=== Helm History (shows every revision) ===")
!helm history my-nginx

In [ ]:
# Step 7: Create your OWN custom Helm chart
# 'helm create' scaffolds the directory structure for a new chart

!helm create my-app

print("=== Chart directory structure ===")
!find my-app -type f | sort

### Understanding the chart structure:

```
my-app/
├── Chart.yaml          # Chart metadata (name, version, description)
├── values.yaml         # Default config values users can override
├── charts/             # Sub-charts (dependencies)
└── templates/          # Kubernetes YAML templates
    ├── deployment.yaml # Template for the Deployment
    ├── service.yaml    # Template for the Service
    ├── _helpers.tpl    # Reusable template snippets
    └── NOTES.txt       # Shown to user after install
```

The `{{ .Values.replicaCount }}` syntax in templates gets replaced with values from `values.yaml` at install time.

In [ ]:
# Peek inside the default values.yaml
!cat my-app/values.yaml

In [ ]:
# Step 8: Lint and template-render the chart before installing
# helm lint checks for errors in your chart
# helm template renders the YAML locally (without hitting the cluster)

print("=== Lint (check for errors) ===")
!helm lint my-app

print("\n=== Rendered YAML (first 60 lines) ===")
!helm template my-app ./my-app | head -60

In [ ]:
# Step 9: Install the custom chart

!helm install my-app ./my-app

print("\n=== All Helm releases ===")
!helm list

In [ ]:
# Step 10: Rollback a release to a previous revision
# This is one of Helm's most powerful features — instant rollback

# First check history of my-nginx
print("=== Release history ===")
!helm history my-nginx

# Rollback to revision 1
print("\nRolling back to revision 1...")
!helm rollback my-nginx 1

print("\n=== History after rollback ===")
!helm history my-nginx

In [ ]:
# Step 11: Uninstall Helm releases
# helm uninstall removes ALL Kubernetes resources that chart created

!helm uninstall my-nginx
!helm uninstall my-app

print("\n=== Remaining Helm releases ===")
!helm list

---
## Cleanup

Delete all resources we created during the assignment.

In [ ]:
# Clean up — delete the Deployment and Service from Part 2 & 3

!kubectl delete deployment nginx-deployment
!kubectl delete service nginx-service

print("\n=== Remaining pods (should be empty shortly) ===")
!kubectl get pods

In [ ]:
# (Optional) Delete the entire kind cluster
# Uncomment the line below if you want to tear down the full cluster

# !kind delete cluster --name k8s-assignment
print("Cluster still running. Uncomment the line above to delete it.")

---
## Summary

| Part | What you did |
|------|--------------|
| **1 — Setup** | Installed `kubectl` + `kind`, created a local K8s cluster, verified nodes |
| **2 — Deployment** | Wrote a YAML manifest, deployed nginx with 2 replicas, exposed it via a Service |
| **3 — Resource Mgmt** | Used `get`, `describe`, `logs`, `scale`, and `delete` on Pods/Deployments/Services |
| **4 — Helm** | Installed Helm, added a repo, deployed a chart, created a custom chart, rolled back a release |

### Key Commands to Remember
```bash
kubectl get pods / deployments / services    # List resources
kubectl describe pod <name>                  # Debug a pod
kubectl apply -f file.yaml                   # Create/update from file
kubectl scale deployment <name> --replicas=N # Scale
kubectl delete deployment <name>             # Delete
helm install <release> <chart>               # Install chart
helm upgrade <release> <chart>               # Upgrade release
helm rollback <release> <revision>           # Rollback
helm uninstall <release>                     # Remove
```